# PCBSegClassNet — Colab Training

End-to-end pipeline on Google Colab GPU: data preprocessing (mask generation + patches + train/val split) → segmentation training → classification training.

**Why Colab?** Local 8 GB GPUs (e.g. RTX 4060 Ti) are too tight for `batch=16` at 512×512 input — the segmentation decoder activation alone is ~4 GB. Colab T4 (16 GB) and above handle it comfortably.

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 is enough; High-RAM not needed).
2. Have `data_raw.zip` ready in Drive at `MyDrive/PCBSegClassNet/data_raw.zip` (~7 GB; contains `pcb_image/` + `smd_annotation/`).
3. Mount Drive when prompted in §4.

## 1. GPU sanity check

In [ ]:
!nvidia-smi

## 2. Clone the repo

If you forked the repo, change the URL to your fork.

In [ ]:
%cd /content
!rm -rf PCBSegClassNet
!git clone -b colab https://github.com/ironmanizawesome/PCBSegClassNet.git
%cd PCBSegClassNet

## 3. Install TF 2.15 + dependencies into Python 3.11

TF 2.15 is the last release on Keras 2 (Keras 3 starts at TF 2.16, which breaks `tf.keras.backend.{dot,transpose}` and other patterns this codebase relies on). But TF 2.15 wheels only target Python 3.9–3.11, while Colab's notebook kernel runs on Python 3.12.

Workaround: Colab images already ship a system `python3.11` binary at `/usr/local/bin/python3.11`. Install TF 2.15 + deps **into that interpreter** and run all training scripts via `!python3.11 ...`. The notebook kernel itself stays on 3.12 — that's fine, we never `import tensorflow` from it.

> 🔑 We can't use `tensorflow[and-cuda]==2.15.0` here — that extra pins `tensorrt-libs==8.6.1`, which is no longer available on PyPI (only 9.x remains). Installing the cudnn / cublas / cuda-runtime / etc. wheels directly is enough; TensorRT is only needed for `tf.experimental.tensorrt` inference, not training.

In [ ]:
# Python 3.11 already exists on Colab; install our stack into it.
!python3.11 -m pip install -q tensorflow==2.15.0 albumentations==1.4.18 opencv-python-headless pyyaml tqdm pandas scikit-learn

# CUDA libs TF needs to dlopen at runtime.
!python3.11 -m pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12 nvidia-cuda-cupti-cu12 nvidia-cuda-nvrtc-cu12 nvidia-cuda-runtime-cu12 nvidia-cufft-cu12 nvidia-curand-cu12 nvidia-cusolver-cu12 nvidia-cusparse-cu12 nvidia-nccl-cu12

In [ ]:
# Verify TF + GPU under Python 3.11 (the interpreter that will actually run training)
!python3.11 -c "import sys, tensorflow as tf; print('Python:', sys.version.split()[0]); print('TF:', tf.__version__); print('Keras:', tf.keras.__version__); print('GPU:', tf.config.list_physical_devices('GPU'))"

## 4. Mount Drive and unpack the raw FPIC archive

This notebook does the **entire data prep pipeline** (mask generation + patches + train/val split) in Colab so you only need to upload the raw FPIC images + annotations (~7 GB) instead of the processed dataset (~18 GB).

### Data layout expected on Drive
Zip the **raw** FPIC images + annotations together:

```
/MyDrive/PCBSegClassNet/
    data_raw.zip                  ← contains: pcb_image/*.png  +  smd_annotation/*.csv
    checkpoints/                  ← (optional, for resume / saved best models)
```

To make the zip on a Windows host:

```powershell
Compress-Archive -Path data\pcb_image, data\smd_annotation -DestinationPath data_raw.zip -Force
```

Why unzip to local disk and not stream from Drive? Drive mounts thousands of small files extremely slowly (API throttling). Always unpack to `/content` for training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
RAW_ZIP = "/content/drive/MyDrive/PCBSegClassNet/data_raw.zip"

import os, time
assert os.path.exists(RAW_ZIP), f"Not found: {RAW_ZIP}"

%cd /content/PCBSegClassNet
!mkdir -p data
t0 = time.time()
!unzip -q -o {RAW_ZIP} -d data/
print(f"Unzip done in {time.time()-t0:.1f}s")

!echo "--- pcb_image:"; ls data/pcb_image/ | wc -l
!echo "--- smd_annotation:"; ls data/smd_annotation/ | wc -l

## 5. Generate masks + classification crops (`create_mask.py`)

Runs through all annotation CSVs, fills polygon masks per component class, and writes:
- `data/segmentation/images/` — HSI + CLAHE preprocessed PCB images
- `data/segmentation/masks/` — RGB masks (color-encoded per class)
- `data/classification/images/<CLASS>/` — individual component crops upscaled with the EDSR super-resolution model in `checkpoints/super_resolution.h5`

GPU-accelerated via the EDSR forward pass. Expect ~10–30 minutes depending on Colab GPU.

In [ ]:
%cd /content/PCBSegClassNet/src/data
!python3.11 create_mask.py \
    -i ../../data/pcb_image/ \
    -a ../../data/smd_annotation/ \
    -id ../../data/segmentation/images \
    -ad ../../data/segmentation/masks \
    -cd ../../data/classification/images/

!echo "--- segmentation/images: $(ls ../../data/segmentation/images 2>/dev/null | wc -l)"
!echo "--- segmentation/masks:  $(ls ../../data/segmentation/masks  2>/dev/null | wc -l)"
!echo "--- classification crops total: $(find ../../data/classification/images -type f | wc -l)"
!echo "--- classification classes:"; ls ../../data/classification/images 2>/dev/null

## 6. Cut 768 px patches and split into train/val (`create_patches.py`)

Cuts the full PCB images + masks into 768×768 patches and moves the patches + classification crops into `train/` and `val/` subfolders (80/20 split). Pure CPU work, ~5 minutes.

After this cell, the dataset layout matches what the training scripts expect:

```
data/segmentation/train/{images,masks}/*.png
data/segmentation/val/{images,masks}/*.png
data/classification/train/<CLASS>/*.png
data/classification/val/<CLASS>/*.png
```

In [ ]:
%cd /content/PCBSegClassNet/src/data
!python3.11 create_patches.py \
    -i ../../data/segmentation/images/ \
    -m ../../data/segmentation/masks \
    -cd ../../data/classification/images/ \
    -ps 768

!echo "--- seg train: $(ls ../../data/segmentation/train/images 2>/dev/null | wc -l) images / $(ls ../../data/segmentation/train/masks 2>/dev/null | wc -l) masks"
!echo "--- seg val:   $(ls ../../data/segmentation/val/images   2>/dev/null | wc -l) images / $(ls ../../data/segmentation/val/masks   2>/dev/null | wc -l) masks"
!echo "--- class train: $(find ../../data/classification/train -type f 2>/dev/null | wc -l) crops"
!echo "--- class val:   $(find ../../data/classification/val   -type f 2>/dev/null | wc -l) crops"

## 7. (Optional) Mirror checkpoints to Drive for persistence

Colab local disk is wiped on session end. Save best model files back to Drive at the end of training. For now, just record the path.

In [ ]:
DRIVE_CKPT_DIR = "/content/drive/MyDrive/PCBSegClassNet/checkpoints"
!mkdir -p {DRIVE_CKPT_DIR}

## 8. Train segmentation

Default config in `cfs/pscn_seg.yml` is `batch_size=16`, `epochs` controlled by `-epoch`.

**First run a 5-epoch sanity pass.** If loss is finite and val_dice_coef is improving, kick off the full 40 epochs.

In [ ]:
# Sanity check: 5 epochs
%cd /content/PCBSegClassNet/src
!python3.11 train_segmentation.py -opt cfs/pscn_seg.yml -epoch 5

In [ ]:
# Full training run (40 epochs)
%cd /content/PCBSegClassNet/src
!python3.11 train_segmentation.py -opt cfs/pscn_seg.yml -epoch 40

In [ ]:
# Backup the best seg checkpoint to Drive
!cp /content/PCBSegClassNet/checkpoints/best_seg.h5 {DRIVE_CKPT_DIR}/best_seg.h5
!ls -la {DRIVE_CKPT_DIR}

## 9. Train classification

In [ ]:
# Sanity check: 5 epochs
%cd /content/PCBSegClassNet/src
!python3.11 train_classification.py -opt cfs/pscn_class.yml -epoch 5

In [ ]:
# Full training run (40 epochs)
%cd /content/PCBSegClassNet/src
!python3.11 train_classification.py -opt cfs/pscn_class.yml -epoch 40

In [ ]:
# Backup the best classification checkpoint to Drive
!cp /content/PCBSegClassNet/checkpoints/best_class.h5 {DRIVE_CKPT_DIR}/best_class.h5
!ls -la {DRIVE_CKPT_DIR}

## 10. (Optional) Evaluate without retraining

Pass `-epoch 0` to skip training; the script will load `best_*.h5` from `checkpoints/` and run `model.evaluate(val_dataset)`. Make sure the checkpoint is in `/content/PCBSegClassNet/checkpoints/` (copy it back from Drive if you reconnected).

In [ ]:
# Restore checkpoints from Drive after a fresh session
!mkdir -p /content/PCBSegClassNet/checkpoints
!cp {DRIVE_CKPT_DIR}/best_seg.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no seg ckpt'
!cp {DRIVE_CKPT_DIR}/best_class.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no class ckpt'

%cd /content/PCBSegClassNet/src
!python3.11 train_segmentation.py -opt cfs/pscn_seg.yml -epoch 0